# Hidden State Extraction

Extracts hidden states from Llama-3.1-8B-Instruct for all generated conversations.

**Design:**
- Layer: last transformer layer (`hidden_states[-1]`)
- Position: last token of each turn (assistant turns only)
- One forward pass per conversation; turn positions found by tokenizing prefixes
- Crescendo: refusal turns filtered out — only extract at turns the target model actually saw
- ActorAttack / X-Teaming: all stored turns used

**Output:**
- `data/representations/hidden_states.npy` — float16, shape (N, 4096)
- `data/representations/metadata.parquet` — one row per (conversation × turn)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE     = "cuda:0"  # cuda:0 because CUDA_VISIBLE_DEVICES=4
DTYPE      = torch.bfloat16

CONV_ROOT  = repo_root / "data" / "conversations"
SAVE_DIR   = repo_root / "data" / "representations"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STATES_PATH   = SAVE_DIR / "hidden_states.npy"
METADATA_PATH = SAVE_DIR / "metadata.parquet"

# Map folder → (framework_label, split_label)
FOLDERS = {
    "crescendo_harmful_v2":   ("crescendo",   "harmful"),
    "actorattack_harmful_v2": ("actorattack", "harmful"),
    "xteaming_harmful_v2":    ("xteaming",    "harmful"),
    # Add benign folders here when generated:
    # "crescendo_benign":   ("crescendo",   "benign"),
    # "actorattack_benign": ("actorattack", "benign"),
    # "xteaming_benign":    ("xteaming",    "benign"),
}

print(f"Device: {DEVICE}")
print(f"Output: {SAVE_DIR}")

In [ ]:
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()
print("Model loaded.")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Hidden dim: {model.config.hidden_size}")

In [ ]:
# ── Refusal detection (Crescendo only) ───────────────────────────────────────
_REFUSAL_PREFIXES = (
    "i cannot", "i can't", "i'm sorry", "i am sorry",
    "i'm unable", "i am unable", "i won't", "i will not",
    "i'm not able", "i am not able", "i'm not going to",
    "i am not going to", "as an ai", "i must decline",
)

def is_refusal(text: str) -> bool:
    t = text.lower().strip()
    return any(t.startswith(p) for p in _REFUSAL_PREFIXES)


def build_context(turns: list[dict], framework: str) -> list[dict]:
    """
    Reconstruct the message list the target model actually saw.
    For Crescendo: skip (user, assistant) pairs where the assistant refused.
    For other frameworks: use all stored turns.
    Returns list of {role, content, turn_idx} dicts for assistant turns to extract at.
    """
    messages = []        # full context (role/content only, for tokenization)
    extract_at = []      # assistant turns to extract hidden states at

    # Group turns into (user, assistant) pairs by turn_idx
    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t["content"]

    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_msg = pair.get("user", "")
        asst_msg = pair.get("assistant", "")

        if not user_msg or not asst_msg:
            continue

        # For Crescendo, skip refusal turns (they were rolled back from target context)
        if framework == "crescendo" and is_refusal(asst_msg):
            continue

        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": asst_msg})
        extract_at.append({"turn_idx": turn_idx, "n_messages": len(messages)})

    return messages, extract_at


def find_turn_end_positions(tokenizer, messages: list[dict], extract_at: list[dict]) -> list[int]:
    """
    For each assistant turn in extract_at, find the index of its last token
    in the full tokenized sequence.
    
    Uses prefix tokenization: tokenize messages[:n_messages] and take the last position.
    Since Llama's chat template is sequential, the prefix length equals the position
    of the last token of that turn in the full sequence.
    """
    positions = []
    for entry in extract_at:
        prefix = messages[:entry["n_messages"]]
        ids = tokenizer.apply_chat_template(
            prefix,
            return_tensors="pt",
            add_generation_prompt=False,
        )
        positions.append(ids.shape[1] - 1)
    return positions


@torch.no_grad()
def extract_hidden_states(model, tokenizer, messages: list[dict], positions: list[int]) -> np.ndarray:
    """
    One forward pass on the full conversation.
    Returns array of shape (len(positions), hidden_dim) in float16.
    """
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=False,
    ).to(model.device)

    outputs = model(input_ids, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_dim)

    vecs = []
    for pos in positions:
        vecs.append(last_hidden[pos].cpu().to(torch.float16).numpy())

    return np.stack(vecs, axis=0)  # (n_turns, hidden_dim)


print("Helpers defined.")

In [ ]:
# ── Main extraction loop ──────────────────────────────────────────────────────
# Resume support: load already-processed (framework, pair_id, attempt) keys
already_done = set()
if METADATA_PATH.exists():
    existing_meta = pd.read_parquet(METADATA_PATH)
    already_done = set(zip(existing_meta["framework"], existing_meta["pair_id"], existing_meta["attempt"]))
    print(f"Resuming: {len(existing_meta)} rows already extracted ({len(already_done)} conversations)")
    all_states  = [np.load(str(STATES_PATH))]  # load existing
    all_meta    = [existing_meta]
else:
    all_states  = []
    all_meta    = []
    print("Starting fresh extraction.")

for folder_name, (framework, split) in FOLDERS.items():
    folder = CONV_ROOT / folder_name
    if not folder.exists():
        print(f"Skipping {folder_name} — folder not found")
        continue

    files = sorted(folder.glob("*.json"))
    print(f"\n{framework}/{split}: {len(files)} files")

    for fpath in tqdm(files, desc=f"{framework}/{split}"):
        conv = json.loads(fpath.read_text())
        pair_id = conv["objective_pair_id"]
        attempt = conv.get("attempt", 1)
        verdict = conv["verdict"]

        if (framework, pair_id, attempt) in already_done:
            continue

        turns = conv.get("turns", [])
        if not turns:
            continue

        try:
            messages, extract_at = build_context(turns, framework)
            if not extract_at:
                continue

            positions = find_turn_end_positions(tokenizer, messages, extract_at)
            states = extract_hidden_states(model, tokenizer, messages, positions)

        except Exception as e:
            tqdm.write(f"  ERROR {fpath.name}: {e}")
            continue

        # Build metadata rows
        rows = []
        for entry, _ in zip(extract_at, positions):
            rows.append({
                "framework":  framework,
                "split":      split,
                "pair_id":    pair_id,
                "attempt":    attempt,
                "turn_idx":   entry["turn_idx"],
                "verdict":    verdict,
                "label":      1 if split == "harmful" else 0,
                "fname":      fpath.name,
            })

        all_states.append(states)
        all_meta.append(pd.DataFrame(rows))
        already_done.add((framework, pair_id, attempt))

print("\nExtraction complete. Saving...")

final_states = np.concatenate(all_states, axis=0).astype(np.float16)
final_meta   = pd.concat(all_meta, ignore_index=True)

np.save(str(STATES_PATH), final_states)
final_meta.to_parquet(METADATA_PATH, index=False)

print(f"Saved {final_states.shape[0]} vectors → {STATES_PATH}")
print(f"  Shape: {final_states.shape}  ({final_states.nbytes / 1e6:.1f} MB)")
print(f"  Metadata: {METADATA_PATH}")
final_meta.groupby(["framework", "split", "verdict"])["turn_idx"].count().rename("n_vectors").reset_index()

## Verification

Sanity checks on the extracted states.

In [ ]:
states = np.load(str(STATES_PATH))
meta   = pd.read_parquet(METADATA_PATH)

print(f"States shape: {states.shape}")
print(f"Metadata rows: {len(meta)}")
print()
print("Vectors per framework/split/verdict:")
display(meta.groupby(["framework", "split", "verdict"])["turn_idx"].count().rename("n_vectors"))
print()
print("Vectors per turn position (across all frameworks):")
display(meta.groupby("turn_idx")["framework"].count().rename("n_vectors"))
print()
print("State stats (sample of 1000):")
sample = states[np.random.choice(len(states), min(1000, len(states)), replace=False)].astype(np.float32)
print(f"  mean={sample.mean():.4f}  std={sample.std():.4f}  min={sample.min():.4f}  max={sample.max():.4f}")
print(f"  NaNs: {np.isnan(sample).sum()}  Infs: {np.isinf(sample).sum()}")